In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Hp\\Desktop\\cifar10-image-classificatio\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Hp\\Desktop\\cifar10-image-classificatio'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    params_image_size: list
    params_batch_size: int

In [6]:
from src.cnnClassifier.constants import *
from src.cnnClassifier.utils.common import read_yaml, create_directories,save_json


In [8]:
from urllib.parse import urlparse
import tensorflow as tf


In [9]:
model = tf.keras.models.load_model(r"C:\Users\Hp\Desktop\cifar10-image-classificatio\artifacts\training\model.h5")

In [15]:


class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_validation_config(self) -> EvaluationConfig:

        eval_config = EvaluationConfig(
            path_of_model=Path(
                r"C:\Users\Hp\Desktop\cifar10-image-classificatio\artifacts\training\model.h5"
            ),

            training_data=Path(
                r"C:\Users\Hp\Desktop\cifar10-image-classificatio\artifacts\data_ingestion"
            ),

            all_params=self.params,

            params_image_size=self.params.IMAGE_SIZE,

            params_batch_size=self.params.BATCH_SIZE,
        )

        return eval_config

In [16]:
import numpy as np
import tensorflow as tf
from pathlib import Path

class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    @staticmethod
    def load_model(path: Path):
        return tf.keras.models.load_model(path)

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)

        X_test = np.load(self.config.training_data / "X_test.npy")
        y_test = np.load(self.config.training_data / "y_test.npy")

        X_test = X_test.astype("float32") / 255.0

        print("X_test Shape:", X_test.shape)
        print("y_test Shape:", y_test.shape)

        self.score = self.model.evaluate(
            X_test,
            y_test,
            batch_size=self.config.params_batch_size,
            verbose=1
        )

    def save_score(self):
        scores = {
            "loss": float(self.score[0]),
            "accuracy": float(self.score[1])
        }

        save_json(
            path=Path("scores.json"),
            data=scores
        )

In [17]:
try:
    config = ConfigurationManager()
    val_config = config.get_validation_config()

    evaluation = Evaluation(val_config)
    evaluation.evaluation()
    evaluation.save_score()

except Exception as e:
    raise e

[2026-07-02 01:16:04,680: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-02 01:16:04,692: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-02 01:16:04,696: INFO: common: created directory at: artifacts]
X_test Shape: (10000, 32, 32, 3)
y_test Shape: (10000, 1)
157/157 [==============================] - 23s 144ms/step - loss: 5.1588 - accuracy: 0.1000
[2026-07-02 01:16:28,776: INFO: common: json file saved at: scores.json]
